# Bootcamp Day 2 — Homework (Pandas + Matplotlib + Seaborn)

**After Day 2 class.** Real datasets: Penguins + Titanic-like.

Vocab allowed: `read_csv`, `head/info/describe`, `value_counts`, `loc/iloc`, boolean filter (`&`, `|`), `groupby`, `fillna`, `dropna`, `drop`, `get_dummies`, `plt.plot/scatter/hist/bar`, `plt.subplots`, `plt.savefig`, `sns.heatmap/boxplot/pairplot`.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/bootcamp_day2_homework.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
print("ready")

---
## Problem 1 — Penguins species report

Load Penguins. Produce a per-species report with these columns:

| species | count | mean_mass_g | max_flipper_mm | min_bill_mm |

Result must be a DataFrame with **3 rows** (one per species) indexed by species name.

In [ ]:
df = sns.load_dataset("penguins").dropna()

# TODO
report = ...

print(report)

In [ ]:
assert report.shape == (3, 4)
assert set(report.index) == {"Adelie", "Chinstrap", "Gentoo"}
assert "count" in report.columns
assert "mean_mass_g" in report.columns
assert int(report.loc["Gentoo", "count"]) == 119
print("Q1 ok")

<details><summary>Hint</summary>

```python
report = df.groupby("species").agg(
    count          = ("body_mass_g", "count"),
    mean_mass_g    = ("body_mass_g", "mean"),
    max_flipper_mm = ("flipper_length_mm", "max"),
    min_bill_mm    = ("bill_length_mm", "min"),
)
```

</details>

---
## Problem 2 — filter + count

Find penguins matching ALL of:
- island is **Biscoe**
- sex is **Female**
- body_mass_g between 4000 and 5500 (inclusive)

1. `subset` — filtered DataFrame
2. `n_match` — count
3. `mean_bill` — average `bill_length_mm` of matching penguins

In [ ]:
df = sns.load_dataset("penguins").dropna()

# TODO
subset    = ...
n_match   = ...
mean_bill = ...

print("n_match:", n_match)
print("mean_bill:", round(mean_bill, 2))

In [ ]:
exp = df[(df["island"]=="Biscoe") & (df["sex"]=="Female") &
         (df["body_mass_g"]>=4000) & (df["body_mass_g"]<=5500)]
assert n_match == len(exp)
assert abs(mean_bill - exp["bill_length_mm"].mean()) < 1e-6
print("Q2 ok")

<details><summary>Hint</summary>

Combine with `&` and parens around every comparison. Or use `.between(lo, hi)` for the range part.

</details>

---
## Problem 3 — clean a messy DataFrame

A messy Titanic-like dataset. Clean it:

1. Drop the column `Notes` entirely
2. Fill `Age` NaN with median age
3. Fill `Embarked` NaN with `"S"`
4. Drop any rows where `Fare` is still NaN
5. Encode `Sex` and `Embarked` to numeric using `pd.get_dummies`

Save final cleaned DataFrame as `clean`.

In [ ]:
df = pd.DataFrame({
    "Name":     [f"P{i}" for i in range(10)],
    "Age":      [22, np.nan, 30, np.nan, 45, 28, 19, np.nan, 35, 50],
    "Fare":     [7.25, 71.83, 8.05, 53.10, 9.50, 30.00, np.nan, 22.0, 45.0, np.nan],
    "Sex":      ["male","female","female","male","male","female","male","female","male","female"],
    "Embarked": ["S","C",np.nan,"S","Q",np.nan,"S","S","C","Q"],
    "Notes":    ["junk"]*10,
    "Survived": [0,1,1,1,0,1,0,1,0,1],
})
print("before:\n", df)

# TODO
clean = ...
print("after:\n", clean)
print("missing:", clean.isna().sum().sum())

In [ ]:
assert "Notes" not in clean.columns
assert clean.isna().sum().sum() == 0
assert "Sex" not in clean.columns, "didn't encode Sex"
assert clean.shape[0] == 8, "should drop 2 rows missing Fare"
print("Q3 ok")

<details><summary>Hint</summary>

```python
clean = (df
    .drop(columns=["Notes"])
    .assign(Age=lambda d: d["Age"].fillna(d["Age"].median()),
            Embarked=lambda d: d["Embarked"].fillna("S"))
    .dropna(subset=["Fare"]))
clean = pd.get_dummies(clean, columns=["Sex", "Embarked"])
```

Or do it step by step without chaining.

</details>

---
## Problem 4 — plot: flipper vs body mass scatter

Plot a **scatter** of `flipper_length_mm` (x) vs `body_mass_g` (y), colored by species.

Requirements:
1. One scatter per species, three colors total
2. Add `xlabel`, `ylabel`, `title`
3. Add `legend()` so species are labeled
4. Save as `flippers.png` BEFORE `plt.show()`

In [ ]:
df = sns.load_dataset("penguins").dropna()

# TODO
plt.figure(figsize=(8, 5))
for species in df["species"].unique():
    ...   # one scatter call per species
...        # labels
...        # title
...        # legend
plt.savefig(...)
plt.show()

In [ ]:
import os
assert os.path.exists("flippers.png"), "didn't save the PNG"
print("Q4 ok — image saved")

<details><summary>Hint</summary>

```python
plt.figure(figsize=(8, 5))
for sp in df["species"].unique():
    sub = df[df["species"] == sp]
    plt.scatter(sub["flipper_length_mm"], sub["body_mass_g"], label=sp, alpha=0.6)
plt.xlabel("flipper length (mm)")
plt.ylabel("body mass (g)")
plt.title("Penguin size by species")
plt.legend()
plt.savefig("flippers.png", dpi=150, bbox_inches="tight")
```

</details>

---
## Problem 5 — capstone: Titanic mini-EDA

Build the Titanic-like dataset below. Produce a **3-panel figure**:

- Panel 1 (left): histogram of `Age` for survived vs not (overlay or side-by-side)
- Panel 2 (middle): bar chart of survival rate by `Pclass`
- Panel 3 (right): seaborn heatmap of numeric column correlations

Then report your 1-sentence finding as the variable `finding`.

In [ ]:
n = 500
df = pd.DataFrame({
    "Pclass":   rng.choice([1,2,3], size=n, p=[0.2, 0.3, 0.5]),
    "Sex":      rng.choice(["male","female"], size=n),
    "Age":      np.where(rng.random(n)>0.1, rng.integers(1,80,n).astype(float), np.nan),
    "Fare":     rng.uniform(5,100,n),
    "Survived": (rng.random(n) > 0.6).astype(int),
})
df["Age"] = df["Age"].fillna(df["Age"].median())

# TODO — 3 panel figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Age hist by Survived
...

# Panel 2: bar chart of survival rate by Pclass
...

# Panel 3: heatmap of numeric correlations
...

plt.tight_layout()
plt.savefig("titanic_mini_eda.png", dpi=150, bbox_inches="tight")
plt.show()

# 1-sentence summary based on what you see
finding = ...   # string

In [ ]:
import os
assert os.path.exists("titanic_mini_eda.png")
assert isinstance(finding, str) and len(finding) > 10, "write a real sentence"
print("Q5 ok — capstone done")

<details><summary>Hint</summary>

```python
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1
axes[0].hist(df[df["Survived"]==1]["Age"], bins=20, alpha=0.6, label="survived")
axes[0].hist(df[df["Survived"]==0]["Age"], bins=20, alpha=0.6, label="not")
axes[0].set_title("Age by Survival"); axes[0].legend()

# Panel 2
rates = df.groupby("Pclass")["Survived"].mean()
axes[1].bar(rates.index.astype(str), rates.values)
axes[1].set_title("Survival rate by Pclass")

# Panel 3
sns.heatmap(df.select_dtypes("number").corr(), annot=True, ax=axes[2], cmap="coolwarm")
axes[2].set_title("Correlations")
```

</details>